In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
fake = pd.read_csv("Fake.csv")
true = pd.read_csv("True.csv")

# Add labels
fake["label"] = 0
true["label"] = 1

# Combine dataset
df = pd.concat([fake, true], axis=0)
df = df.sample(frac=1).reset_index(drop=True)  # shuffle

df

,title,text,subject,date,label
0,DOUBLE AMPUTEE VET Blasts Obama’s War Strategy...,War veteran Johnny Joey Jones praised Presid...,politics,"Apr 15, 2017",0
1,"Australia, NZ officials discuss screening for ...",WELLINGTON (Reuters) - New Zealand and Austral...,worldnews,"November 20, 2017",1
2,Democratic leader vows fair nomination process...,WASHINGTON (Reuters) - The head of the Democra...,politicsNews,"November 5, 2017",1
3,BREAKING: HOUSE Votes “Yes” On American Securi...,As the House moves closer to actually represen...,politics,"Nov 19, 2015",0
4,Shelling across Pakistan-India border kills si...,ISLAMABAD (Reuters) - Shelling along the dispu...,worldnews,"September 22, 2017",1
...,...,...,...,...,...
44893,India's muted response to Trump's Jerusalem mo...,NEW DELHI (Reuters) - A dozen Arab ambassadors...,worldnews,"December 18, 2017",1
44894,Trump's tweet on London train bombing just spe...,LONDON (Reuters) - Britain s interior minister...,worldnews,"September 17, 2017",1
44895,TPP leaders' meeting postponed after Canada di...,"DANANG, Vietnam (Reuters) - A planned meeting ...",worldnews,"November 10, 2017",1
44896,South Korea's Moon first suggested Trump visit...,SEOUL (Reuters) - South Korean President Moon ...,worldnews,"November 8, 2017",1


In [3]:
# Combine title + text
df["content"] = df["title"] + " " + df["text"]

# Select features
X = df["content"]
y = df["label"]

In [4]:
vocab_size = 5000
max_len = 200

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X)

X_seq = tokenizer.texts_to_sequences(X)
X_pad = pad_sequences(X_seq, maxlen=max_len, padding='post')

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_pad, y, test_size=0.2, random_state=42
)

In [6]:
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

C:\Users\anami\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding (Embedding)                │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ ?                           │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ ?                           │     0 (unbuilt) │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ ?                           │     0 (unbuilt) │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.8981 - loss: 0.2449 - val_accuracy: 0.9470 - val_loss: 0.1563
Epoch 2/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.9763 - loss: 0.0851 - val_accuracy: 0.9825 - val_loss: 0.0655
Epoch 3/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.9268 - loss: 0.1792 - val_accuracy: 0.9723 - val_loss: 0.0842
Epoch 4/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 71ms/step - accuracy: 0.9816 - loss: 0.0615 - val_accuracy: 0.9866 - val_loss: 0.0517
Epoch 5/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 71ms/step - accuracy: 0.9906 - loss: 0.0328 - val_accuracy: 0.9864 - val_loss: 0.0510
Epoch 6/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 71ms/step - accuracy: 0.9910 - loss: 0.0293 - val_accuracy: 0.9777 - val_loss: 0.0785
Epoch 7/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 71ms/step - accuracy: 0.9945 - loss: 0.0191 - val_accuracy: 0.9873 - val_loss: 0.0594
Epoch 8/10
562/562 ━━━━━━━━━━━━━━━━━━━━ 41s 70ms/step - accuracy: 0.9975 - loss: 0.0096 - 

In [8]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

281/281 ━━━━━━━━━━━━━━━━━━━━ 5s 18ms/step
Accuracy: 0.9896436525612472
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4807
           1       0.99      0.99      0.99      4173

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [15]:
def predict_news(text):
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len, padding='post')
    pred = model.predict(padded)[0][0]
    
    if pred > 0.5:
        return "Real News"
    else:
        return "Fake News"

# Example
predict_news("The earth will stop rotating for 10 seconds tomorrow causing global chaos.")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step


'Fake News'

In [16]:
# save the model
model.save("fake_news_model.h5")

In [17]:
import pickle

with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)